In [ ]:
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py pyyaml

In [ ]:
!apt-get install -y -q git-lfs
!git lfs install
!git clone https://github.com/noshou/APS360.git /kaggle/working/APS360

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/APS360")

from ScatterNet.config import RunConfig, DEFAULT_BUCKETS
from train import main

cfg = RunConfig(

    # --- paths ---
    hdf5           = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5",  # HDF5 dataset
    db             = "/kaggle/working/APS360/Preprocess/scatternet",       # SQLite encoding stem
    ckpt_best      = "/kaggle/working/scatternet_best.pt",                 # saved on val improvement
    ckpt_resume    = "/kaggle/working/scatternet_resume.pt",               # saved every epoch
    metrics        = "/kaggle/working/scatternet_metrics.json",            # per-epoch loss + R2
    resume         = None,                                                  # path to resume from, or None

    # --- model ---
    lambda_1       = 128,   # atom embedding dimension
    lambda_2       = 5,     # message passing rounds
    lambda_3       = 128,   # OutputHead hidden width
    lambda_4       = 4,     # MLP halving steps (2^lambda_4 <= lambda_3)
    lambda_5       = 256,   # Random Fourier Features
    msg_seed       = 42,    # RFF frequency matrix seed
    msg_chunk      = 256,   # atoms per M-chunk; lower = less GPU memory, slower
    eps_embd       = 1e-8,  # numerical floor in Embed
    eps_msgp       = 1e-3,  # numerical floor in MessagePass

    # --- loss ---
    lambda_6       = 0.1,   # form-factor penalty weight
    lambda_7       = 0.1,   # sigma inverse-L1 regularisation weight
    eps_sigma      = 1e-4,  # floor added to sigma before inverse-L1 penalty (prevents 1/sigma -> inf)

    # --- training ---
    lr             = 3e-4,  # Adam learning rate
    weight_decay   = 1e-5,  # Adam L2 weight decay
    grad_clip      = 1.0,   # max gradient norm
    epochs         = 50,    # epochs to train
    batcher_seed   = 0,     # train/val/test split seed
    atom_size_ceil = -1,    # max atoms per batch (-1 = auto: 3x largest molecule)
    num_workers    = 3,     # DataLoader workers
    max_batches    = None,  # cap batches per epoch (None = no limit)
    use_amp        = True,  # mixed precision: halves activation memory via float16 forw

    # --- data ---
    buckets        = DEFAULT_BUCKETS,  # (min_atoms, max_atoms) size buckets to include
)

main(cfg)